In [ ]:
import os
from pathlib import Path

import sys
sys.path.append('./coeqwalpackage')

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
from coeqwalpackage.metrics import add_water_year_column, create_subset_unit, read_in_df
from coeqwalpackage import cqwlutils as cu
from coeqwalpackage import plotting as pu

In [ ]:
pu.enable_headless_mode()

## Define control file name


In [ ]:
CtrlFile = 'CalSim3DataExtractionInitFile_v4.xlsx'
CtrlTab = 'Init'

## Read from control file


In [ ]:
ScenarioListFile, ScenarioListTab, ScenarioListPath, DVDssNamesOutPath, SVDssNamesOutPath, ScenarioIndicesOutPath, DssDirsOutPath, VarListPath, VarListFile, VarListTab, VarOutPath, DataOutPath, ConvertDataOutPath, ExtractionSubPath, DemandDeliverySubPath, ModelSubPath, GroupDataDirPath, ScenarioDir, DVDssMin, DVDssMax, SVDssMin, SVDssMax, NameMin, NameMax, DirMin, DirMax, IndexMin, IndexMax, StartMin, StartMax, EndMin, EndMax, VarMin, VarMax, DemandFilePath, DemandFileName, DemandFileTab, DemMin, DemMax, InflowOutSubPath, InflowFilePath, InflowFileName, InflowFileTab, InflowMin, InflowMax = cu.read_init_file(CtrlFile, CtrlTab)

## Notebook configuration


In [ ]:
SCENARIO_SETS = [
    {"name": "s20_s11_s21_s23_s24_s47_s51", "baseline": 20, "compare": [11, 21, 24, 23, 47, 51]},
    {"name": "s20_s25_s26_s27_s28_s62", "baseline": 20, "compare": [25, 26, 27, 28, 62]},
    {"name": "s20_s29_s30_s31_s32_s33_s46", "baseline": 20, "compare": [30, 29, 32, 31, 33, 46]},
    {"name": "s20_s39_s40_s41_s42", "baseline": 20, "compare": [40, 41, 42, 39]},
    {"name": "s20_s44_s45_s65", "baseline": 20, "compare": [44, 45, 65]}]

# TUCP FILTERING MODE
# "baseline" - All scenarios in a set use the baseline's TUCP years
# "per_scenario" - Each scenario uses its own TUCP years
TUCP_MODE = "baseline"

LABEL_STYLE = "label_only"
BASELINE_LABEL_OVERRIDE = None
DPI = 300

wyt_wet = [1, 2, 3]
wyt_dry = [4, 5]
month = 5

TUCP_VAR_BASE = "TUCP_TRIGGER_DV"

### Read in Data


In [ ]:
df, dss_names = read_in_df(ConvertDataOutPath, DVDssNamesOutPath)
df = add_water_year_column(df)

### Scenario metadata


In [ ]:
scenario_names = cu.build_scenario_labels_from_listing(ScenarioListPath, ScenarioListFile, ScenarioListTab)

all_scenarios = set()
for scenario_set in SCENARIO_SETS:
    all_scenarios.add(scenario_set["baseline"])
    all_scenarios.update(scenario_set["compare"])

tucp_years_by_scenario = {}
scenarios_with_tucp = set()  # Track which scenarios have valid TUCP data

for sid in sorted(all_scenarios):
    try:
        years = cu.selected_tucp_years(df, scenario=sid, tucp_var_base=TUCP_VAR_BASE)
        tucp_years_by_scenario[sid] = years
        scenarios_with_tucp.add(sid)
        print(f"Scenario s{sid:04d}: TUCP years -> {years}")
    except KeyError as e:
        tucp_years_by_scenario[sid] = None 
        print(f"Scenario s{sid:04d}: ⚠No TUCP data available - will skip TUCP plots for this scenario")

print(f"\nTUCP_MODE = '{TUCP_MODE}'")
print(f"Scenarios with valid TUCP data: {sorted(scenarios_with_tucp)}")

In [ ]:
plots_root = Path(GroupDataDirPath) / "plots_output"
plots_root.mkdir(parents=True, exist_ok=True)
print(f"Output root: {plots_root}")

In [ ]:
variables = [
    "S_MELON_", "S_SHSTA_", "S_OROVL_", "S_TRNTY_", "S_FOLSM_",
    "S_SLUIS_s", "DEL_CVP_PAG_N", "DEL_CVP_PAG_S",
    "DEL_CVP_PSC_N", "DEL_CVP_PEX_S", "DEL_SWP_PMI_S",
    "DEL_SWP_TOTA_", "DEL_SWP_PAG_N", 
    "C_SAC000_s", "C_SAC041_s", "C_SAC085_s", "C_SAC122_s", 
    "SP_SAC083_YBP037", "C_SJR070_s", "C_DMC000_TD_s", "C_CAA003_TD_s",
    "X2_PRV_KM_", "EM_EC_MONTH_", "RS_EC_MONTH_", "JP_EC_MONTH_",
    "AWOANN_ALL_DV", "SG_SACAB", "SG_SACBB", "SG_SACFB", "SG_SACAMR", "SG_SACBASIN"]

UNITS_MAP = {"X2_PRV_KM_": "KM", "EM_EC_MONTH_": "UMHOS/CM", "RS_EC_MONTH_": "UMHOS/CM", "JP_EC_MONTH_": "UMHOS/CM"}

STORAGE_VARS = ["S_MELON_", "S_SHSTA_", "S_OROVL_", "S_TRNTY_", "S_FOLSM_", "S_SLUIS_s"]

In [ ]:
def plots_per_var(var: str) -> int:
    if var == "AWOANN_ALL_DV":
        return 3
    cnt = 13  # base 9 + 4 (exceed_oct, ann_tot, ann_exceed_all, ann_exceed_tucp)
    if var in STORAGE_VARS:
        cnt += 2  # sept and apr exceedance
    return cnt

active_vars = []
for v in variables:
    _units = pu.infer_units(v, UNITS_MAP, default="TAF")
    subset = create_subset_unit(df, v, _units)
    if not subset.empty:
        active_vars.append(v)

plots_per_set = sum(plots_per_var(v) for v in active_vars)
normal_total = plots_per_set * len(SCENARIO_SETS)
parallel_per_set = 6 * max(len(s["compare"]) for s in SCENARIO_SETS)
parallel_total = parallel_per_set * len(SCENARIO_SETS)

print(f"Active variables: {len(active_vars)}")
print(f"Plots per set: {plots_per_set}")
print(f"Scenario sets: {len(SCENARIO_SETS)}")
print(f"Total standard plots: {normal_total}")
print(f"Total parallel plots: ~{parallel_total}")

### Plotting loop

In [ ]:
failed_variables = []

print("Starting Plotting Loop...")
print(f"Processing {len(SCENARIO_SETS)} scenario set(s)")
print(f"TUCP_MODE = '{TUCP_MODE}'")
print("=" * 50)

# Outer loop over scenario sets
for set_idx, scenario_set in enumerate(SCENARIO_SETS):
    set_name = scenario_set["name"]
    baseline = scenario_set["baseline"]
    compare = scenario_set["compare"]
    scenarios = [baseline] + compare

    print(f"\n[Set {set_idx + 1}/{len(SCENARIO_SETS)}] {set_name}")
    print(f"  Baseline: s{baseline:04d}, Compare: {['s' + str(s).zfill(4) for s in compare]}")

    # --- Check if all scenarios have TUCP data ---
    set_has_tucp = all(tucp_years_by_scenario.get(s) is not None for s in scenarios)
    if not set_has_tucp:
        missing = [f"s{s:04d}" for s in scenarios if tucp_years_by_scenario.get(s) is None]
        print(f"  ⚠️ TUCP data missing for: {missing} - TUCP plots will be SKIPPED for this set")

    # --- TUCP Mode Logic ---
    if set_has_tucp:
        if TUCP_MODE == "baseline":
            baseline_tucp_years = tucp_years_by_scenario.get(baseline, [])
            tucp_years_for_set = {s: baseline_tucp_years for s in scenarios}
            print(f"  TUCP years (baseline mode): {baseline_tucp_years}")
        else:  # "per_scenario"
            tucp_years_for_set = {s: tucp_years_by_scenario.get(s, []) for s in scenarios}
            print(f"  TUCP years (per_scenario mode): varying per scenario")
    else:
        tucp_years_for_set = {s: [] for s in scenarios}

    # --- Create output directories for this set ---
    set_root = plots_root / set_name
    set_root.mkdir(parents=True, exist_ok=True)

    exceedance_dir = set_root / "exceedance"
    moy_dir = set_root / "moy_avg"
    ts_dir = set_root / "mon_ts"
    ann_exceed_dir = set_root / "ann_exceed"
    ann_tot_dir = set_root / "ann_tot"

    for d in [exceedance_dir, moy_dir, ts_dir, ann_exceed_dir, ann_tot_dir]:
        d.mkdir(parents=True, exist_ok=True)

    # --- Plotting loop for this set ---
    with tqdm(total=plots_per_set, desc=f"Plotting: {set_name}", unit="fig", mininterval=0.5) as pbar:
        for var in active_vars:
            try:
                units = pu.infer_units(var, UNITS_MAP, default="TAF")
                var_df = create_subset_unit(df, var, units)
                if var_df.empty:
                    print(f"WARNING: No data found in df for var={var} with units={units}. Skipping.")
                    continue

                # Common args for all _multi functions
                common_plot_args = dict(
                    scenarios=scenarios,
                    scenario_labels=scenario_names,
                    baseline_id=baseline,
                    baseline_label_override=BASELINE_LABEL_OVERRIDE,
                    label_style=LABEL_STYLE,
                    dpi=DPI
                )
                # Additional args for functions supporting TUCP filtering
                tucp_args = dict(tucp_var_base=TUCP_VAR_BASE, tucp_years=tucp_years_for_set)

                # Use set-specific directories
                exceed_path = str(exceedance_dir)
                moy_path = str(moy_dir)
                ts_path = str(ts_dir)
                ann_exceed_path = str(ann_exceed_dir)
                ann_tot_path = str(ann_tot_dir)

                if var != "AWOANN_ALL_DV":
                    # Standard Exceedance
                    pu.plot_exceedance_multi(df, varname=var, units=units, xLab='Probability', pTitle="Exceedance Probability (All Years)", lTitle='Scenarios', fTitle="exceed_all", fPath=exceed_path, **common_plot_args, **tucp_args)
                    pbar.update(1)

                    # TUCP - only if set has TUCP data
                    if set_has_tucp:
                        pu.plot_exceedance_multi(df, varname=var, units=units, use_tucp=True, xLab='Probability', pTitle="Exceedance Probability (TUCP Years)", lTitle='Scenarios', fTitle="exceed_tucp", fPath=exceed_path, **common_plot_args, **tucp_args)
                    pbar.update(1)

                    # WYT
                    pu.plot_exceedance_multi(df, varname=var, units=units, use_wyt=True, wyt=wyt_wet, wyt_month=month, xLab='Probability', pTitle=f"Exceedance Probability for Water Year Types {wyt_wet}", fTitle="exceed_wet", lTitle='Scenarios', fPath=exceed_path, **common_plot_args, **tucp_args)
                    pbar.update(1)
                    pu.plot_exceedance_multi(df, varname=var, units=units, use_wyt=True, wyt=wyt_dry, wyt_month=month, xLab='Probability', pTitle=f"Exceedance Probability for Water Year Types {wyt_dry}", fTitle="exceed_dry", lTitle='Scenarios', fPath=exceed_path, **common_plot_args, **tucp_args)
                    pbar.update(1)

                    # October exceedance
                    pu.plot_exceedance_multi(df, varname=var, units=units, months=[10], xLab='Probability', pTitle="Exceedance Probability (October)", lTitle='Scenarios', fTitle="exceed_oct", fPath=exceed_path, **common_plot_args, **tucp_args)
                    pbar.update(1)

                    # MOY
                    pu.plot_moy_averages_multi(df, varname=var, units=units, xLab="Month", pTitle="MOY Averages (All Years)", lTitle='Scenarios', fTitle="moy_all", fPath=moy_path, **common_plot_args, **tucp_args)
                    pbar.update(1)
                    if set_has_tucp:
                        pu.plot_moy_averages_multi(df, varname=var, units=units, use_tucp=True, xLab="Month", pTitle="MOY Averages (TUCP Years)", lTitle='Scenarios', fTitle="moy_tucp", fPath=moy_path, **common_plot_args, **tucp_args)
                    pbar.update(1)
                    pu.plot_moy_averages_multi(df, varname=var, units=units, use_wyt=True, wyt=wyt_wet, wyt_month=month, xLab="Month", pTitle=f"MOY Averages for Water Year Types {wyt_wet}", lTitle='Scenarios', fTitle="moy_wet", fPath=moy_path, **common_plot_args, **tucp_args)
                    pbar.update(1)
                    pu.plot_moy_averages_multi(df, varname=var, units=units, use_wyt=True, wyt=wyt_dry, wyt_month=month, xLab="Month", pTitle=f"MOY Averages for Water Year Types {wyt_dry}", lTitle='Scenarios', fTitle="moy_dry", fPath=moy_path, **common_plot_args, **tucp_args)
                    pbar.update(1)

                    # TS
                    pu.plot_ts_multi(df, varname=var, units=units, pTitle="Monthly Time Series", xLab="Date", lTitle='Scenarios', fTitle="mon_ts", fPath=ts_path, **common_plot_args)
                    pbar.update(1)

                    # Annual totals
                    pu.plot_annual_totals_ts_multi(df, varname=var, units=units, xLab="Water Year", pTitle="Annual Totals", lTitle='Scenarios', fTitle="ann_tot", fPath=ann_tot_path, **common_plot_args)
                    pbar.update(1)

                    # Annual exceedance
                    pu.annualize_exceedance_multi(df, varname=var, units=units, pTitle="Annual Exceedance", xLab="Exceedance Probability", lTitle='Scenarios', fTitle="ann_exceed_all", fPath=ann_exceed_path, **common_plot_args, **tucp_args)
                    pbar.update(1)
                    if set_has_tucp:
                        pu.annualize_exceedance_multi(df, varname=var, units=units, use_tucp=True, pTitle="Annual Exceedance (TUCP Years)", xLab="Exceedance Probability", lTitle='Scenarios', fTitle="ann_exceed_tucp", fPath=ann_exceed_path, **common_plot_args, **tucp_args)
                    pbar.update(1)

                if var in STORAGE_VARS:
                    pu.annualize_exceedance_multi(df, varname=var, units=units, months=[9], pTitle="Annual Exceedance (Sept Only)", xLab="Exceedance Probability", lTitle='Scenarios', fTitle="ann_exceed_sept", fPath=ann_exceed_path, **common_plot_args, **tucp_args)
                    pbar.update(1)
                    pu.annualize_exceedance_multi(df, varname=var, units=units, months=[4], pTitle="Annual Exceedance (April Only)", xLab="Exceedance Probability", lTitle='Scenarios', fTitle="ann_exceed_apr", fPath=ann_exceed_path, **common_plot_args, **tucp_args)
                    pbar.update(1)
                    if set_has_tucp:
                        pu.annualize_exceedance_multi(df, varname=var, units=units, use_tucp=True, months=[9], pTitle="Annual Exceedance (Sept TUCP Only)", xLab="Exceedance Probability", lTitle='Scenarios', fTitle="ann_exceed_sept_tucp", fPath=ann_exceed_path, **common_plot_args, **tucp_args)
                        pbar.update(1)
                        pu.annualize_exceedance_multi(df, varname=var, units=units, use_tucp=True, months=[4], pTitle="Annual Exceedance (April TUCP Only)", xLab="Exceedance Probability", lTitle='Scenarios', fTitle="ann_exceed_apr_tucp", fPath=ann_exceed_path, **common_plot_args, **tucp_args)
                        pbar.update(1)
                    else:
                        pbar.update(2)  # Skip count for missing TUCP plots

                if var == "AWOANN_ALL_DV":
                    pu.annualize_exceedance_multi(df, varname=var, units=units, months=[3], pTitle="Annual Exceedance (March Only)", xLab="Exceedance Probability", lTitle='Scenarios', fTitle="ann_exceed_mar", fPath=ann_exceed_path, **common_plot_args, **tucp_args)
                    pbar.update(1)
                    pu.annualize_exceedance_multi(df, varname=var, units=units, use_wyt=True, wyt=wyt_wet, wyt_month=month, months=[3], pTitle=f"Annual Exceedance (March Only) for Water Year Types {wyt_wet}", xLab="Exceedance Probability", lTitle='Scenarios', fTitle="ann_exceed_mar_wet", fPath=ann_exceed_path, **common_plot_args, **tucp_args)
                    pbar.update(1)
                    pu.annualize_exceedance_multi(df, varname=var, units=units, use_wyt=True, wyt=wyt_dry, wyt_month=month, months=[3], pTitle=f"Annual Exceedance (March Only) for Water Year Types {wyt_dry}", xLab="Exceedance Probability", lTitle='Scenarios', fTitle="ann_exceed_mar_dry", fPath=ann_exceed_path, **common_plot_args, **tucp_args)
                    pbar.update(1)

            except Exception as e:
                failed_variables.append(f"{set_name}/{var} (Error: {str(e)})")
                pbar.write(f"!! SKIPPING {var}: {str(e)}")
                continue

    print(f"  Completed: {set_name}")

print("\n" + "=" * 50)
print(f"Processing Complete. {len(failed_variables)} variables failed.")
if failed_variables:
    print("Failed Variables:")
    for f in failed_variables:
        print(f"  - {f}")
print("=" * 50)

### Parallel Line Plots

In [ ]:
axis_label_map = {"Sac Valley Ag Deliveries": {"calsim_vars": ["DEL_NOD_AG_"], "units": "TAF", "months": None},
                  "SJ Valley Ag Deliveries": {"calsim_vars": ["DEL_SOD_AG_"], "units": "TAF", "months": None},
                  "Sac Valley Municipal Deliveries": {"calsim_vars": ["DEL_NOD_MI_"], "units": "TAF", "months": None},
                  "SoCal Municipal Deliveries": {"calsim_vars": ["DEL_SOD_MI_"], "units": "TAF", "months": None},
                  "Delta Exports": {"calsim_vars": ["TOTAL_EXPORTS_"], "units": "TAF", "months": None},
                  "Delta Outflows": {"calsim_vars": ["NDO_"], "units": "TAF", "months": None},
                  "Sac River Inflows": {"calsim_vars": ["C_SAC041_"], "units": "TAF", "months": None},
                  "SJ River Inflows": {"calsim_vars": ["C_SJR070_"], "units": "TAF", "months": None},
                  "X2 Salinity (Apr)": {"calsim_vars": ["X2_PRV_KM_"], "units": "KM", "months": [5]},
                  "X2 Salinity (Oct)": {"calsim_vars": ["X2_PRV_KM_"], "units": "KM", "months": [11]},
                  "North of Delta Storage (Sep)": {"calsim_vars": ["NOD_STORAGE_"], "units": "TAF","months": [9]},
                  "South of Delta Storage (Sep)": {"calsim_vars": ["SOD_STORAGE_"], "units": "TAF", "months": [9]}}

In [ ]:
# Base directory for parallel plots - per-set subdirs created in loop
parallel_plots_root = Path(GroupDataDirPath) / "parallel_plots_output"
parallel_plots_root.mkdir(parents=True, exist_ok=True)
print(f"Parallel plots root: {parallel_plots_root}")

In [ ]:
print("Starting Parallel Plots...")
print(f"TUCP_MODE = '{TUCP_MODE}'")
print("=" * 50)

# Outer loop over scenario sets
for set_idx, scenario_set in enumerate(SCENARIO_SETS):
    set_name = scenario_set["name"]
    baseline = scenario_set["baseline"]
    compare = scenario_set["compare"]

    print(f"\n[Set {set_idx + 1}/{len(SCENARIO_SETS)}] {set_name}")

    # Create output directory for this set's parallel plots
    set_parallel_path = parallel_plots_root / set_name
    set_parallel_path.mkdir(parents=True, exist_ok=True)

    # Generate comparison pairs: baseline vs each compare scenario
    scenario_comps = [(baseline, s) for s in compare]

    # --- Check if all scenarios have TUCP data ---
    all_scenarios_in_set = [baseline] + compare
    set_has_tucp = all(tucp_years_by_scenario.get(s) is not None for s in all_scenarios_in_set)
    if not set_has_tucp:
        missing = [f"s{s:04d}" for s in all_scenarios_in_set if tucp_years_by_scenario.get(s) is None]
        print(f"  ⚠️ TUCP data missing for: {missing} - TUCP parallel plots will be SKIPPED")

    # --- TUCP Mode Logic ---
    if set_has_tucp:
        if TUCP_MODE == "baseline":
            baseline_tucp_years = tucp_years_by_scenario.get(baseline, [])
            tucp_years_for_set = {s: baseline_tucp_years for s in all_scenarios_in_set}
        else:  # "per_scenario"
            tucp_years_for_set = {s: tucp_years_by_scenario.get(s, []) for s in all_scenarios_in_set}
    else:
        tucp_years_for_set = {s: [] for s in all_scenarios_in_set}

    # In per_scenario mode, skip relative TUCP plots (nonsensical: % change between different year sets)
    skip_relative_tucp = (TUCP_MODE == "per_scenario") or not set_has_tucp
    if skip_relative_tucp and set_has_tucp:
        print("  NOTE: Skipping relative TUCP plots in per_scenario mode")
    
    # Calculate plot count
    if not set_has_tucp:
        parallel_count = 3 * len(scenario_comps)  # Only 3 all-years plots per pair
    elif skip_relative_tucp:
        parallel_count = 4 * len(scenario_comps)  # 3 all-years + 1 absolute TUCP
    else:
        parallel_count = 6 * len(scenario_comps)

    with tqdm(total=parallel_count, desc=f"Parallel: {set_name}", unit="fig", mininterval=0.5) as pb2:
        for (s1, s2) in scenario_comps:
            # Get TUCP years based on mode
            s1_tucp = tucp_years_for_set.get(s1, [])
            s2_tucp = tucp_years_for_set.get(s2, [])

            # Get scenario styles for parallel plot highlighting
            style_dict = pu.get_scenario_styles([s1, s2], scenario_labels=scenario_names, baseline_id=baseline)
            highlight_colors = [style_dict[s1]['color'], style_dict[s2]['color']]
            highlight_descs = [style_dict[s1]['label'], style_dict[s2]['label']]

            # --- 1. Absolute values (All Years) ---
            abs_all_df = pu.build_parallel_df_absolute(df, axis_label_map, scenario1=s1, scenario2=s2, subset_years_s1=None, subset_years_s2=None)
            fig, ax = pu.custom_parallel_coordinates_highlight_scenarios(
                objs=abs_all_df, columns_axes=abs_all_df.columns, axis_labels=abs_all_df.columns,
                ideal_direction='top', minmaxs=['max'] * len(abs_all_df.columns),
                highlight_indices=[f"Scen{s1}", f"Scen{s2}"], highlight_colors=highlight_colors,
                highlight_descriptions=highlight_descs,
                title=f"Absolute Mean Values (All Years): s{s1} vs. s{s2}",
                figsize=(22, 8), fontsize=12,
                save_fig_filename=str(set_parallel_path / f"parallel_abs_all_s{s1}_s{s2}.png"),
                dpi=DPI
            )
            plt.close(fig)
            pb2.update(1)

            # --- 2. Relative % Difference (All Years) ---
            rel_all_df = pu.build_parallel_df_relative(df, axis_label_map, scenario1=s1, scenario2=s2, subset_years_s1=None, subset_years_s2=None)
            fig, ax = pu.custom_parallel_coordinates_highlight_scenarios_baseline_at_zero(
                objs=rel_all_df, columns_axes=rel_all_df.columns, axis_labels=rel_all_df.columns,
                highlight_indices=[f"Scen{s1}", f"Scen{s2}"],
                highlight_colors=[highlight_colors[0], highlight_colors[1]],
                highlight_descriptions=[highlight_descs[0], highlight_descs[1]],
                title=f"Relative % Difference (All Years): s{s1} vs. s{s2}",
                figsize=(22, 8), fontsize=12,
                save_fig_filename=str(set_parallel_path / f"parallel_rel_all_s{s1}_s{s2}.png"),
                dpi=DPI
            )
            plt.close(fig)
            pb2.update(1)

            # --- 3. Relative % Diff + Baseline Values (All Years) ---
            abs_base_all = pu.build_parallel_df_absolute(df, axis_label_map, scenario1=s1, scenario2=s1, subset_years_s1=None, subset_years_s2=None)
            baseline_abs_all = abs_base_all.iloc[[0]]
            fig, ax = pu.custom_parallel_coordinates_relative_with_baseline_values(
                objs_rel=rel_all_df, baseline_abs=baseline_abs_all, axis_label_map=axis_label_map,
                columns_axes=rel_all_df.columns, axis_labels=rel_all_df.columns,
                alpha_base=0.8, lw_base=1.5, fontsize=12, figsize=(22, 8),
                save_fig_filename=str(set_parallel_path / f"parallel_rel_all_s{s1}_s{s2}_withBaselineVals.png"),
                title=f"Relative % Diff (All Years) + Baseline Values: s{s1} vs. s{s2}",
                highlight_indices=[f"Scen{s1}", f"Scen{s2}"],
                highlight_colors=[highlight_colors[0], highlight_colors[1]],
                highlight_descriptions=[highlight_descs[0], highlight_descs[1]],
                dpi=DPI
            )
            plt.close(fig)
            pb2.update(1)

            # --- 4, 5, 6: TUCP plots - only if set has TUCP data ---
            if set_has_tucp:
                # --- 4. Absolute values (TUCP Years) ---
                abs_tucp_df = pu.build_parallel_df_absolute(df, axis_label_map, scenario1=s1, scenario2=s2, subset_years_s1=s1_tucp, subset_years_s2=s2_tucp)
                fig, ax = pu.custom_parallel_coordinates_highlight_scenarios(
                    objs=abs_tucp_df, columns_axes=abs_tucp_df.columns, axis_labels=abs_tucp_df.columns,
                    ideal_direction='top', minmaxs=['max'] * len(abs_tucp_df.columns),
                    highlight_indices=[f"Scen{s1}", f"Scen{s2}"], highlight_colors=highlight_colors,
                    highlight_descriptions=highlight_descs,
                    title=f"Absolute Mean Values (TUCP Years): s{s1} vs. s{s2}",
                    figsize=(22, 8), fontsize=12,
                    save_fig_filename=str(set_parallel_path / f"parallel_abs_tucp_s{s1}_s{s2}.png"),
                    dpi=DPI
                )
                plt.close(fig)
                pb2.update(1)

                # --- 5 & 6: Relative TUCP plots (SKIP in per_scenario mode) ---
                if not skip_relative_tucp:
                    # --- 5. Relative % Difference (TUCP Years) ---
                    rel_tucp_df = pu.build_parallel_df_relative(df, axis_label_map, scenario1=s1, scenario2=s2, subset_years_s1=s1_tucp, subset_years_s2=s2_tucp)
                    fig, ax = pu.custom_parallel_coordinates_highlight_scenarios_baseline_at_zero(
                        objs=rel_tucp_df, columns_axes=rel_tucp_df.columns, axis_labels=rel_tucp_df.columns,
                        highlight_indices=[f"Scen{s1}", f"Scen{s2}"],
                        highlight_colors=[highlight_colors[0], highlight_colors[1]],
                        highlight_descriptions=[highlight_descs[0], highlight_descs[1]],
                        title=f"Relative % Difference (TUCP Years): s{s1} vs. s{s2}",
                        figsize=(22, 8), fontsize=12,
                        save_fig_filename=str(set_parallel_path / f"parallel_rel_tucp_s{s1}_s{s2}.png"),
                        dpi=DPI
                    )
                    plt.close(fig)
                    pb2.update(1)

                    # --- 6. Relative % Diff + Baseline Values (TUCP Years) ---
                    abs_base_tucp = pu.build_parallel_df_absolute(df, axis_label_map, scenario1=s1, scenario2=s1, subset_years_s1=s1_tucp, subset_years_s2=s1_tucp)
                    baseline_abs_tucp = abs_base_tucp.iloc[[0]]
                    fig, ax = pu.custom_parallel_coordinates_relative_with_baseline_values(
                        objs_rel=rel_tucp_df, baseline_abs=baseline_abs_tucp, axis_label_map=axis_label_map,
                        columns_axes=rel_tucp_df.columns, axis_labels=rel_tucp_df.columns,
                        alpha_base=0.8, lw_base=1.5, fontsize=12, figsize=(22, 8),
                        save_fig_filename=str(set_parallel_path / f"parallel_rel_tucp_s{s1}_s{s2}_withBaselineVals.png"),
                        title=f"Relative % Diff (TUCP) + Baseline Values: s{s1} vs. s{s2}",
                        highlight_indices=[f"Scen{s1}", f"Scen{s2}"],
                        highlight_colors=[highlight_colors[0], highlight_colors[1]],
                        highlight_descriptions=[highlight_descs[0], highlight_descs[1]],
                        dpi=DPI
                    )
                    plt.close(fig)
                    pb2.update(1)

            print(f"  Finished parallel plots for s{s1} vs. s{s2}")

    print(f"  Completed: {set_name}")

print("\n" + "=" * 50)
print("Parallel Plots Complete.")
print("=" * 50)